# spatialautocorrelation_test
Sanitized research code. Default input is SYNTHETIC, not paper data. Run via tools/run_models.py. See docs/MANUSCRIPT_ALIGNMENT.md.


In [ ]:
import os
from pathlib import Path
import importlib.util
import warnings

os.environ.setdefault('MPLCONFIGDIR', '/tmp/matplotlib-codex')
Path(os.environ['MPLCONFIGDIR']).mkdir(parents=True, exist_ok=True)

required = {
    'numpy': 'numpy',
    'pandas': 'pandas',
    'geopandas': 'geopandas',
    'matplotlib': 'matplotlib',
    'libpysal': 'libpysal',
    'esda': 'esda',
}
missing = [pkg for pkg, module in required.items() if importlib.util.find_spec(module) is None]
if missing:
    raise ImportError(
        'Missing required packages: ' + ', '.join(missing) +
        '. Run this notebook with the geospatial Python environment.'
    )

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm, ListedColormap, TwoSlopeNorm
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

from libpysal.weights import Queen, KNN
from libpysal.weights.spatial_lag import lag_spatial
from esda.moran import Moran, Moran_Local

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', message='.*The weights matrix is not fully connected.*')

# Public package: never search parent folders for private data.
import os
from IPython.display import display
PROJECT_ROOT = Path(os.environ['UFUG_PUBLIC_ROOT']).resolve()
DATA_PATH = Path(os.environ.get('UFUG_MODEL_DATA', str(PROJECT_ROOT / 'data/synthetic/model_cells.geojson'))).resolve()
DATA_RELATIVE = DATA_PATH
OUTPUT_DIR = Path(os.environ.get('UFUG_OUTPUT_ROOT', str(PROJECT_ROOT / 'outputs'))) / "spatialautocorrelation_test"
if DATA_PATH != (PROJECT_ROOT / 'data/synthetic/model_cells.geojson').resolve() and OUTPUT_DIR.resolve().is_relative_to(PROJECT_ROOT):
    raise ValueError('Private model outputs must resolve outside Public, including redirected model directories')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


PERMUTATIONS = 999
RANDOM_SEED = 20260419
LISA_ALPHA = 0.05
KNN_FALLBACK_K = 8

PARK_ORDER = ['amk', 'jurong', 'punggol']
PARK_LABELS = {'amk': 'AMK', 'jurong': 'Jurong', 'punggol': 'Punggol'}
OUTCOMES = {
    'log_F_h1': {
        'source': 'F_h',
        'label': 'Visit frequency',
        'axis_label': 'log(visit frequency + 1)',
        'stem': 'log_visit_frequency',
    },
    'log_D_h1': {
        'source': 'D_h',
        'label': 'Visit duration',
        'axis_label': 'log(visit duration + 1)',
        'stem': 'log_visit_duration',
    },
}

mpl.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 600,
    'savefig.bbox': 'tight',
    'font.family': 'DejaVu Sans',
    'font.size': 9,
    'axes.titlesize': 10,
    'axes.labelsize': 9,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
    'axes.linewidth': 0.7,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

print(f'Project root: {PROJECT_ROOT}')
print(f'Data: {DATA_PATH.name}')
print(f'Outputs: {OUTPUT_DIR.name}')


In [ ]:
gdf = gpd.read_file(DATA_PATH)

if gdf.crs is None:
    gdf = gdf.set_crs(3414)
if gdf.crs.to_epsg() != 3414:
    gdf = gdf.to_crs(3414)

gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy()
gdf['park'] = gdf['park'].astype(str).str.lower()
gdf = gdf[gdf['park'].isin(PARK_ORDER)].copy()
gdf['park'] = pd.Categorical(gdf['park'], categories=PARK_ORDER, ordered=True)

for col in ['F_h', 'D_h']:
    gdf[col] = pd.to_numeric(gdf[col], errors='coerce').fillna(0)

gdf['log_F_h1'] = np.log1p(gdf['F_h'])
gdf['log_D_h1'] = np.log1p(gdf['D_h'])
gdf['visited'] = (gdf['F_h'] > 0).astype(int)
gdf['positive_duration'] = (gdf['D_h'] > 0).astype(int)

gdf = gdf.sort_values(['park', 'grid_id']).reset_index(drop=True)

summary = (
    gdf.groupby('park', observed=True)
    .agg(
        n_hex=('grid_id', 'count'),
        total_visits=('F_h', 'sum'),
        mean_visits=('F_h', 'mean'),
        share_visited=('visited', 'mean'),
        total_duration=('D_h', 'sum'),
        mean_duration=('D_h', 'mean'),
        share_positive_duration=('positive_duration', 'mean'),
    )
    .reset_index()
)
summary['park'] = summary['park'].map(PARK_LABELS)
summary[['share_visited', 'share_positive_duration']] = summary[['share_visited', 'share_positive_duration']].round(3)
summary[['mean_visits', 'mean_duration']] = summary[['mean_visits', 'mean_duration']].round(2)

display(summary)
print(f'Loaded {len(gdf):,} hexagons across {gdf["park"].nunique()} parks.')
print(f'CRS: {gdf.crs}')


In [ ]:
def zscore(values):
    arr = np.asarray(values, dtype=float)
    sd = np.nanstd(arr)
    if not np.isfinite(sd) or sd == 0:
        return np.zeros_like(arr)
    return (arr - np.nanmean(arr)) / sd


def bh_fdr(p_values):
    p = np.asarray(p_values, dtype=float)
    q = np.full(p.shape, np.nan, dtype=float)
    valid = np.isfinite(p)
    if valid.sum() == 0:
        return q
    pv = p[valid]
    order = np.argsort(pv)
    ranked = pv[order]
    n = len(ranked)
    adjusted = ranked * n / np.arange(1, n + 1)
    adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]
    adjusted = np.clip(adjusted, 0, 1)
    q_valid = np.empty_like(adjusted)
    q_valid[order] = adjusted
    q[valid] = q_valid
    return q


def build_weights(sub_gdf, k=KNN_FALLBACK_K):
    queen = Queen.from_dataframe(sub_gdf, use_index=True)
    if len(queen.islands) == 0 and getattr(queen, 'n_components', 1) == 1:
        queen.transform = 'R'
        return queen, 'Queen contiguity'

    k = min(k, max(len(sub_gdf) - 1, 1))
    knn = KNN.from_dataframe(sub_gdf, k=k)
    knn.transform = 'R'
    return knn, f'KNN, k={k}'


LISA_ORDER = ['High-high', 'Low-low', 'High-low', 'Low-high', 'Not significant']
LISA_COLORS = {
    'High-high': '#b2182b',
    'Low-low': '#2166ac',
    'High-low': '#ef8a62',
    'Low-high': '#67a9cf',
    'Not significant': '#f0f0f0',
}
QUADRANT_LABELS = {
    1: 'High-high',
    2: 'Low-high',
    3: 'Low-low',
    4: 'High-low',
}


def lisa_cluster_labels(local_moran, alpha=LISA_ALPHA):
    labels = np.array([QUADRANT_LABELS.get(int(q), 'Not significant') for q in local_moran.q], dtype=object)
    labels = np.where(local_moran.p_sim <= alpha, labels, 'Not significant')
    return labels


def p_stars(p_value):
    if p_value < 0.001:
        return '***'
    if p_value < 0.01:
        return '**'
    if p_value < 0.05:
        return '*'
    return ''


def save_figure(fig, stem):
    paths = []
    for ext in ['png', 'svg']:
        path = OUTPUT_DIR / f'{stem}.{ext}'
        fig.savefig(path, dpi=600, bbox_inches='tight', facecolor='white')
        paths.append(path)
    return paths

def show_or_close(fig):
    backend = mpl.get_backend().lower()
    if 'agg' in backend and 'inline' not in backend:
        plt.close(fig)
    else:
        show_or_close(fig)


In [ ]:
results = {}
global_rows = []
cluster_rows = []
local_gdf = gdf.copy()

for outcome_col in OUTCOMES:
    local_gdf[f'{outcome_col}_lisa_cluster'] = 'Not computed'
    local_gdf[f'{outcome_col}_lisa_I'] = np.nan
    local_gdf[f'{outcome_col}_lisa_p'] = np.nan
    local_gdf[f'{outcome_col}_lisa_q'] = np.nan
    local_gdf[f'{outcome_col}_lisa_p_fdr'] = np.nan
    local_gdf[f'{outcome_col}_z'] = np.nan
    local_gdf[f'{outcome_col}_spatial_lag_z'] = np.nan

for park in PARK_ORDER:
    sub = gdf[gdf['park'] == park].copy().reset_index().rename(columns={'index': 'source_index'})
    w, weight_label = build_weights(sub)
    neighbor_counts = np.array(list(w.cardinalities.values()))

    for outcome_col, meta in OUTCOMES.items():
        y = sub[outcome_col].to_numpy(dtype=float)
        z = zscore(y)
        wz = lag_spatial(w, z)

        np.random.seed(RANDOM_SEED)
        global_moran = Moran(y, w, permutations=PERMUTATIONS)
        local_moran = Moran_Local(
            y,
            w,
            permutations=PERMUTATIONS,
            seed=RANDOM_SEED,
            n_jobs=1,
            keep_simulations=False,
        )
        labels = lisa_cluster_labels(local_moran)
        p_fdr = bh_fdr(local_moran.p_sim)

        source_index = sub['source_index'].to_numpy()
        local_gdf.loc[source_index, f'{outcome_col}_lisa_cluster'] = labels
        local_gdf.loc[source_index, f'{outcome_col}_lisa_I'] = local_moran.Is
        local_gdf.loc[source_index, f'{outcome_col}_lisa_p'] = local_moran.p_sim
        local_gdf.loc[source_index, f'{outcome_col}_lisa_q'] = local_moran.q
        local_gdf.loc[source_index, f'{outcome_col}_lisa_p_fdr'] = p_fdr
        local_gdf.loc[source_index, f'{outcome_col}_z'] = z
        local_gdf.loc[source_index, f'{outcome_col}_spatial_lag_z'] = wz

        counts = pd.Series(labels).value_counts().reindex(LISA_ORDER, fill_value=0)
        for cluster, count in counts.items():
            cluster_rows.append({
                'park': park,
                'park_label': PARK_LABELS[park],
                'outcome': outcome_col,
                'outcome_label': meta['label'],
                'cluster': cluster,
                'n_hex': int(count),
                'share_hex': float(count / len(sub)),
            })

        global_rows.append({
            'park': park,
            'park_label': PARK_LABELS[park],
            'outcome': outcome_col,
            'outcome_label': meta['label'],
            'n_hex': int(len(sub)),
            'weights': weight_label,
            'min_neighbors': int(neighbor_counts.min()),
            'mean_neighbors': float(neighbor_counts.mean()),
            'max_neighbors': int(neighbor_counts.max()),
            'moran_I': float(global_moran.I),
            'expected_I': float(global_moran.EI),
            'z_norm': float(global_moran.z_norm),
            'p_norm': float(global_moran.p_norm),
            'p_sim': float(global_moran.p_sim),
            'permutations': int(global_moran.permutations),
            'lisa_high_high': int(counts['High-high']),
            'lisa_low_low': int(counts['Low-low']),
            'lisa_high_low': int(counts['High-low']),
            'lisa_low_high': int(counts['Low-high']),
        })

        results[(park, outcome_col)] = {
            'data': sub,
            'weights': w,
            'weights_label': weight_label,
            'global_moran': global_moran,
            'local_moran': local_moran,
            'labels': labels,
            'p_fdr': p_fdr,
            'z': z,
            'spatial_lag_z': wz,
        }

global_moran_table = pd.DataFrame(global_rows)
cluster_counts = pd.DataFrame(cluster_rows)

global_moran_report = global_moran_table.copy()
for col in ['mean_neighbors', 'moran_I', 'expected_I', 'z_norm', 'p_norm', 'p_sim']:
    global_moran_report[col] = global_moran_report[col].round(4)
cluster_counts_report = cluster_counts.copy()
cluster_counts_report['share_hex'] = cluster_counts_report['share_hex'].round(3)

display(global_moran_report[[
    'park_label', 'outcome_label', 'n_hex', 'weights', 'moran_I', 'z_norm', 'p_norm', 'p_sim',
    'lisa_high_high', 'lisa_low_low', 'lisa_high_low', 'lisa_low_high'
]])
display(cluster_counts_report)


In [ ]:
def nice_scale_length(target_length):
    if target_length <= 0 or not np.isfinite(target_length):
        return 100
    magnitude = 10 ** np.floor(np.log10(target_length))
    for step in [1, 2, 5, 10]:
        candidate = step * magnitude
        if candidate >= target_length:
            return int(candidate)
    return int(10 * magnitude)


def add_scale_bar(ax, fraction=0.22):
    x0, x1 = ax.get_xlim()
    y0, y1 = ax.get_ylim()
    width = x1 - x0
    height = y1 - y0
    length = nice_scale_length(width * fraction)
    xb = x0 + width * 0.07
    yb = y0 + height * 0.08
    ax.plot([xb, xb + length], [yb, yb], color='black', linewidth=1.4, solid_capstyle='butt', zorder=5)
    ax.text(xb + length / 2, yb + height * 0.025, f'{length:g} m', ha='center', va='bottom', fontsize=7)


def add_north_arrow(ax):
    ax.annotate(
        'N',
        xy=(0.92, 0.93),
        xytext=(0.92, 0.80),
        xycoords='axes fraction',
        textcoords='axes fraction',
        ha='center',
        va='center',
        fontsize=8,
        arrowprops=dict(arrowstyle='-|>', linewidth=0.8, color='black'),
    )


def format_map_axis(ax):
    ax.set_axis_off()
    ax.set_aspect('equal')
    add_scale_bar(ax)
    add_north_arrow(ax)


In [ ]:
fig, ax = plt.subplots(figsize=(6.8, 3.0))
plot_table = global_moran_table.copy()
plot_table['park_label'] = pd.Categorical(plot_table['park_label'], [PARK_LABELS[p] for p in PARK_ORDER], ordered=True)
plot_table['outcome_label'] = pd.Categorical(
    plot_table['outcome_label'],
    [meta['label'] for meta in OUTCOMES.values()],
    ordered=True,
)
pivot_i = plot_table.pivot(index='park_label', columns='outcome_label', values='moran_I')
pivot_p = plot_table.pivot(index='park_label', columns='outcome_label', values='p_sim')

max_abs = max(abs(pivot_i.min().min()), abs(pivot_i.max().max()), 0.05)
norm = TwoSlopeNorm(vmin=-max_abs, vcenter=0, vmax=max_abs)
im = ax.imshow(pivot_i.to_numpy(), cmap='RdBu_r', norm=norm)

ax.set_xticks(np.arange(pivot_i.shape[1]), labels=pivot_i.columns)
ax.set_yticks(np.arange(pivot_i.shape[0]), labels=pivot_i.index)
ax.tick_params(length=0)

for i, park_label in enumerate(pivot_i.index):
    for j, outcome_label in enumerate(pivot_i.columns):
        value = pivot_i.loc[park_label, outcome_label]
        p_value = pivot_p.loc[park_label, outcome_label]
        ax.text(j, i, f'I={value:.3f}{p_stars(p_value)}\np={p_value:.3f}', ha='center', va='center', fontsize=8)

for spine in ax.spines.values():
    spine.set_visible(False)

cbar = fig.colorbar(im, ax=ax, shrink=0.78, pad=0.03)
cbar.set_label("Moran's I")
ax.set_title("Global spatial autocorrelation by park and outcome")
fig.tight_layout()
save_figure(fig, 'global_moran_summary')
show_or_close(fig)


In [ ]:
for outcome_col, meta in OUTCOMES.items():
    fig, axes = plt.subplots(1, len(PARK_ORDER), figsize=(9.8, 3.2), sharex=True, sharey=True)
    all_z = np.concatenate([results[(park, outcome_col)]['z'] for park in PARK_ORDER])
    all_lag = np.concatenate([results[(park, outcome_col)]['spatial_lag_z'] for park in PARK_ORDER])
    lim = max(abs(np.nanmin(all_z)), abs(np.nanmax(all_z)), abs(np.nanmin(all_lag)), abs(np.nanmax(all_lag)), 1)
    lim = np.ceil(lim * 10) / 10

    for ax, park in zip(axes, PARK_ORDER):
        res = results[(park, outcome_col)]
        z = res['z']
        wz = res['spatial_lag_z']
        moran = res['global_moran']

        ax.scatter(z, wz, s=22, color='#2f6f9f', edgecolor='white', linewidth=0.35, alpha=0.88)
        x_line = np.array([-lim, lim])
        ax.plot(x_line, moran.I * x_line, color='#b2182b', linewidth=1.2)
        ax.axhline(0, color='0.35', linewidth=0.7, linestyle=':')
        ax.axvline(0, color='0.35', linewidth=0.7, linestyle=':')
        ax.set_xlim(-lim, lim)
        ax.set_ylim(-lim, lim)
        ax.set_title(f'{PARK_LABELS[park]}\nI={moran.I:.3f}{p_stars(moran.p_sim)}, p={moran.p_sim:.3f}')
        ax.grid(True, color='0.90', linewidth=0.5)
        ax.set_xlabel(f'Standardized {meta["axis_label"]}')

    axes[0].set_ylabel('Spatial lag')
    fig.suptitle(f"Moran scatterplots: {meta['label']}", y=1.02, fontsize=11)
    fig.tight_layout()
    save_figure(fig, f'moran_scatter_{meta["stem"]}')
    show_or_close(fig)


In [ ]:
legend_handles = [
    Patch(facecolor=LISA_COLORS[label], edgecolor='0.55', linewidth=0.35, label=label)
    for label in LISA_ORDER
]

for outcome_col, meta in OUTCOMES.items():
    fig, axes = plt.subplots(1, len(PARK_ORDER), figsize=(10.8, 3.9))

    for ax, park in zip(axes, PARK_ORDER):
        source_index = results[(park, outcome_col)]['data']['source_index'].to_numpy()
        plot_gdf = local_gdf.loc[source_index].copy()
        cluster_col = f'{outcome_col}_lisa_cluster'
        colors = plot_gdf[cluster_col].map(LISA_COLORS)
        plot_gdf.plot(ax=ax, color=colors, edgecolor='white', linewidth=0.22)

        counts = plot_gdf[cluster_col].value_counts()
        hh = int(counts.get('High-high', 0))
        ll = int(counts.get('Low-low', 0))
        ax.set_title(f'{PARK_LABELS[park]}\nHH={hh}, LL={ll}')
        format_map_axis(ax)

    fig.legend(handles=legend_handles, loc='lower center', ncol=5, frameon=False, bbox_to_anchor=(0.5, -0.01))
    fig.suptitle(f"Local Moran/LISA clusters: {meta['label']} (p <= {LISA_ALPHA})", y=1.02, fontsize=11)
    fig.subplots_adjust(bottom=0.16, wspace=0.03)
    save_figure(fig, f'lisa_clusters_{meta["stem"]}')
    show_or_close(fig)


In [ ]:
for outcome_col, meta in OUTCOMES.items():
    fig, axes = plt.subplots(1, len(PARK_ORDER), figsize=(10.8, 3.9))
    values = gdf[outcome_col].to_numpy(dtype=float)
    vmin = float(np.nanmin(values))
    vmax = float(np.nanquantile(values, 0.98))
    if vmax <= vmin:
        vmax = float(np.nanmax(values))
    norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)
    cmap = mpl.colormaps['viridis']

    for ax, park in zip(axes, PARK_ORDER):
        plot_gdf = gdf[gdf['park'] == park].copy()
        plot_gdf.plot(
            ax=ax,
            column=outcome_col,
            cmap=cmap,
            norm=norm,
            edgecolor='white',
            linewidth=0.22,
            legend=False,
        )
        ax.set_title(PARK_LABELS[park])
        format_map_axis(ax)

    sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=axes.ravel().tolist(), shrink=0.72, pad=0.02)
    cbar.set_label(meta['axis_label'])
    fig.suptitle(f"Spatial pattern of {meta['label'].lower()}", y=1.02, fontsize=11)
    fig.subplots_adjust(wspace=0.03)
    save_figure(fig, f'spatial_pattern_{meta["stem"]}')
    show_or_close(fig)


In [ ]:
global_csv = OUTPUT_DIR / 'global_moran_summary.csv'
cluster_csv = OUTPUT_DIR / 'lisa_cluster_counts.csv'
local_geojson = OUTPUT_DIR / 'lisa_clusters_all.geojson'

global_moran_table.to_csv(global_csv, index=False)
cluster_counts.to_csv(cluster_csv, index=False)

local_export = local_gdf.copy()
for col in local_export.select_dtypes(include=['category']).columns:
    local_export[col] = local_export[col].astype(str)
local_export.to_file(local_geojson, driver='GeoJSON')

exported_files = sorted(p.name for p in OUTPUT_DIR.iterdir() if p.is_file())
display(pd.DataFrame({'output_file': exported_files}))
print(f'Wrote {len(exported_files)} files to {OUTPUT_DIR.name}')
